## Imports

Here import all crucial packages etc.

In [ ]:
# Code here

In [ ]:
import json
import os
import pandas as pd
from transformers import (DistilBertTokenizer, DistilBertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from sklearn.metrics import f1_score, precision_score, recall_score
import torch
from transformers import EvalPrediction, pipeline

## Utils

Helper functions that you will use

In [ ]:
#Code here

In [ ]:
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
class DisinformationDataset(torch.utils.data.Dataset):
    """
    This class wraps our tokenized data and labels so PyTorch can easily loop through them during training. It converts each input into tensors and returns them with the label — all in the format the model expects.
    """
    # When we create an instance of dataset, we pass in encodings and labels
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    # This method tells PyTorch how to get one item (input + label).
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    # Returns how many examples are in the dataset (needed by DataLoader).
    def __len__(self):
        return len(self.labels)


def load_and_process_data(file_path: str, label_column: str = "label") -> pd.DataFrame:
    """
    Loads the data from a CSV file and processes the labels.
    Args:
        file_path (str): Path to the CSV file.
        label_column (str): The column name containing the labels.
        text_column (str): The column name containing the text content.
    Returns:
        pd.DataFrame: Processed dataframe with labels and text content.
    """
    data = pd.read_csv(file_path, encoding='utf-8')
    data[label_column] = data[label_column].apply(lambda x: 1 if "fake" in x.lower() else 0)
    return data


def save_metrics_to_json(metrics: dict, output_file_path: str):
    """
    Saves the metrics to a JSON file.
    Args:
        metrics (dict): The evaluation metrics.
        output_file_path (str): The file path to save the metrics.
    """
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    with open(output_file_path, 'w') as output_file:
        json.dump(metrics, output_file, indent=4)

In [ ]:
def compute_metrics(pred=None, y_true=None, y_pred=None):
    """
    Computes F1 scores (micro, macro, weighted) for both training and testing data.

    If `pred` is provided, it computes metrics for the trainer using `EvalPrediction`.
    If `y_true` and `y_pred` are provided, it computes metrics for test data predictions.

    Parameters:
        - pred (EvalPrediction, optional): The evaluation prediction object for Trainer.
        - y_true (list, optional): The ground truth labels for the test data.
        - y_pred (list, optional): The predicted labels for the test data.

    Returns:
        - dict: A dictionary containing F1 metrics.
    """
    if pred is not None:
        # When working with the Trainer, pred is an EvalPrediction object
        labels = pred.label_ids
        y_pred = pred.predictions.argmax(-1)
    elif y_true is not None and y_pred is not None:
        # If y_true and y_pred are provided, use them for test evaluation
        labels = y_true
    else:
        raise ValueError("Either `pred` or both `y_true` and `y_pred` must be provided.")

        # Compute F1 scores
    f1 = f1_score(y_true=labels, y_pred=y_pred)
    precision = precision_score(y_true=labels, y_pred=y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true=labels, y_pred=y_pred, average='binary', zero_division=0)



    return {
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def compute_metrics_for_trainer(pred: EvalPrediction):
    return compute_metrics(pred=pred)

# Assignment

# Fine-Tuning BERT Model to Fake News detection

## Import Train, Validation and Test data

Import all datasets and load and preprocess train and validation

Link to direcotry with data: https://github.com/ArkadiusDS/NLP-Labs/tree/master/data/CoAID/

In [ ]:
# Define the URLs pointing to the raw CSV data files hosted on GitHub.

url_test = 'https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/test.csv'
url_train = 'https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/train.csv'
url_valid = 'https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/validation.csv'

# Download the datasets from GitHub using the wget command-line tool.
# Each file is saved with a simple filename for ease of use.

!wget -O test.csv {url_test}
!wget -O train.csv {url_train}
!wget -O validation.csv {url_valid}

--2025-05-23 09:10:28--  https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 221757 (217K) [text/plain]
Saving to: ‘test.csv’

test.csv            100%[===================>] 216.56K  --.-KB/s    in 0.02s   

2025-05-23 09:10:28 (9.04 MB/s) - ‘test.csv’ saved [221757/221757]

--2025-05-23 09:10:28--  https://raw.githubusercontent.com/ArkadiusDS/NLP-Labs/refs/heads/master/data/CoAID/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 155653

In [ ]:
# Load and preprocess the datasets using the custom function 'load_and_process_data'
# This function will load the CSV data files, process the labels, and return the data in a usable dataframe format.

# Load and process the training data
train_data = load_and_process_data('train.csv')

# Load and process the validation data
validation_data = load_and_process_data('validation.csv')

## Load model and tokenizer

Firstly create two dicts id2label and label2id and then load model and tokenizer
Then use well-known distilled version of BERT model for faster fine-tuning: 'distilbert/distilbert-base-uncased' or any other model you wish.

In [ ]:
id2label = {0: "Real", 1: "Fake"}
label2id = {"Real": 0, "Fake": 1}

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert/distilbert-base-uncased', num_labels=2, id2label=id2label, label2id=label2id)


## Tokenize datasets and prepare it for fine-tuning

You may use DisinformationDataset class for data preparation.

In [ ]:
# Tokenize the datasets (training and validation) to prepare them for input into the BERT model.
# Tokenization converts the raw text data into a format the BERT model can process.

# Tokenizing the training dataset
train_encodings = tokenizer(
        train_data['content'].tolist(),
        truncation=True,
        padding=True,
        max_length=256
    )

# Tokenizing the validation dataset
val_encodings = tokenizer(
        validation_data['content'].tolist(),
        truncation=True,
        padding=True,
        max_length=256
    )

In [ ]:
# Create custom datasets for training and validation using the DisinformationDataset class.
# These datasets will format the tokenized text data and corresponding labels into a format that can be used by the model during training and evaluation.

# Create the training dataset: it combines the tokenized training data and corresponding labels
train_dataset = DisinformationDataset(train_encodings, train_data['label'].tolist())

# Create the validation dataset: it combines the tokenized validation data and corresponding labels
val_dataset = DisinformationDataset(val_encodings, validation_data['label'].tolist())

## Fine-tune BERT model on at least 3 sets of hyperparameters

Check F1 score, precision and recall for each fine-tuned model and at the end choose set of hyperparameters that gives you best results. For each set of hyperparameters write down the final metrics. You need to acheive at least below result on validation dataset:

"f1": 0.91,
"recall": 0.91,
"precision": 0.91

Remember you need to achieve these minimum results on VALIDATION dataset and the best model on validation dataset will have to be used for predictions on test dataset.


In [ ]:
hyperparameter_sets=[
    {
        "output_dir":"./results",
        "eval_strategy":"steps",
        "learning_rate":5e-5,
        "per_device_train_batch_size":16,
        "per_device_eval_batch_size":16,
        "num_train_epochs":3,
        "warmup_ratio":0.06,
        "weight_decay":0.01,
        "fp16":True,
        "metric_for_best_model":"f1",
        "load_best_model_at_end":True,
        "save_total_limit":2,
        "greater_is_better":True,
        "save_strategy":"steps",
        "eval_steps":100,
        "save_on_each_node":True,
        "report_to":[],
    },
    {
        "output_dir":"./results",
        "eval_strategy":"steps",
        "learning_rate":3e-5,
        "per_device_train_batch_size":32,
        "per_device_eval_batch_size":32,
        "num_train_epochs":4,
        "warmup_ratio":0.06,
        "weight_decay":0.005,
        "fp16":True,
        "metric_for_best_model":"f1",
        "load_best_model_at_end":True,
        "save_total_limit":2,
        "greater_is_better":True,
        "save_strategy":"steps",
        "eval_steps":100,
        "save_on_each_node":True,
        "report_to":[],
        "logging_steps":100,
    },
    {
        "output_dir":"./results",
        "eval_strategy":"steps",
        "learning_rate":2e-5,
        "per_device_train_batch_size":16,
        "per_device_eval_batch_size":16,
        "num_train_epochs":5,
        "warmup_ratio":0.06,
        "weight_decay":0.01,
        "fp16":True,
        "metric_for_best_model":"f1",
        "load_best_model_at_end":True,
        "save_total_limit":2,
        "greater_is_better":True,
        "save_strategy":"steps",
        "eval_steps":100,
        "save_on_each_node":True,
        "report_to":[],
        "logging_steps":100,
    },
]

In [ ]:
best_f1_on_validation=-1.0
best_hyperparameters=None
best_trainer=None

all_hp_results=[]

for i, params in enumerate(hyperparameter_sets):
    print(f"\n--- Running Hyperparameter Set {i}/{len(hyperparameter_sets)} ---")

    current_model=AutoModelForSequenceClassification.from_pretrained(
        'distilbert/distilbert-base-uncased',
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id
    )

    training_args=TrainingArguments(
        output_dir=params["output_dir"],
        eval_strategy=params["eval_strategy"],
        learning_rate=params["learning_rate"],
        per_device_train_batch_size=params["per_device_train_batch_size"],
        per_device_eval_batch_size=params["per_device_eval_batch_size"],
        num_train_epochs=params["num_train_epochs"],
        warmup_ratio=params["warmup_ratio"],
        weight_decay=params["weight_decay"],
        fp16=params["fp16"],
        metric_for_best_model=params["metric_for_best_model"],
        load_best_model_at_end=params["load_best_model_at_end"],
        save_total_limit=params["save_total_limit"],
        greater_is_better=params["greater_is_better"],
        save_strategy=params["save_strategy"],
        eval_steps=params["eval_steps"],
        save_on_each_node=params["save_on_each_node"],
        report_to=params["report_to"],
        logging_steps=params["logging_steps"],
    )

    trainer0Trainer(
        model=current_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_for_trainer,
    )

    trainer.train()

    validation_metrics=trainer.evaluate()

    f1=validation_metrics.get("eval_f1", 0.0)
    precision=validation_metrics.get("eval_precision", 0.0)
    recall=validation_metrics.get("eval_recall", 0.0)

    current_hp_results={
        "hyperparameters":params,
        "validation_metrics":{
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "accuracy": validation_metrics.get("eval_accuracy", 0.0),
            "loss": validation_metrics.get("eval_loss", 0.0)
        }
    }
    all_hp_results.append(current_hp_results)

    print(f"\nResults for HP Set {i} on Validation:")
    print(f"  F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

    if (f1 >= 0.91 and precision >= 0.91 and recall >= 0.91 and f1 > best_f1_on_validation):
        best_f1_on_validation=f1
        best_hyperparameters=params
        best_trainer=trainer
        best_experimenti=i

In [ ]:
print("best hyperpameters: ", best_hyperparameters)
print("best f1 on vlaidation", best_f1_on_validation)
print("best trainer", best_trainer)
print("best experiment:", best_experimenti)

best hyperpameters:  {'output_dir': './results', 'eval_strategy': 'steps', 'learning_rate': 2e-05, 'per_device_train_batch_size': 16, 'per_device_eval_batch_size': 16, 'num_train_epochs': 5, 'warmup_ratio': 0.06, 'weight_decay': 0.01, 'fp16': True, 'metric_for_best_model': 'f1', 'load_best_model_at_end': True, 'save_total_limit': 2, 'greater_is_better': True, 'save_strategy': 'steps', 'eval_steps': 100, 'save_on_each_node': True, 'report_to': [], 'logging_steps': 100}
best f1 on vlaidation 0.9552238805970149
best trainer <transformers.trainer.Trainer object at 0x7f73a71202d0>
best experiment: 2


In [ ]:
model_saved_path='output/final/'
trainer.save_model(model_saved_path)
tokenizer.save_pretrained(model_saved_path)

('output/final/tokenizer_config.json',
 'output/final/special_tokens_map.json',
 'output/final/vocab.txt',
 'output/final/added_tokens.json')

## Final prediction on test dataset

Take best model and hyperparameters on validation and predict on test dataset. Compute evaluation metrics f1, precision and recall.

In [ ]:
# Load the test data and preprocess
test_data = load_and_process_data('test.csv')

In [ ]:
# Load the pipeline with CUDA
classifier = pipeline(
    task="text-classification",
    model=model_saved_path,
    tokenizer=model_saved_path,
    device=0,
    truncation=True,
    padding=True,
    max_length=256
)

results = classifier(test_data["content"].tolist(), batch_size=32)
test_data["predictions"] = [1 if r["label"] == "Fake" else 0 for r in results]

# Compute evaluation metrics on the test data
evaluation_results = compute_metrics(y_true=test_data["label"], y_pred=test_data["predictions"])
evaluation_results

Device set to use cuda:0


{'f1': 0.975609756097561,
 'precision': 0.9803921568627451,
 'recall': 0.970873786407767}

# Final file with results and description

In [ ]:
import json

All keys in your dictionary have to be the same as below. The only changes you should do in terms of keys is changing names of hyperparameters, e.g. instead of key "name_of_hyperparameter_0" if you used learning rate then write "learning_rate". Other important information in the dictionary below and comments. Each value says what is expected.

Example dictionary provided under the template.

Template for your structured resulting file

In [ ]:
data = {
    # Everything in experiment_0 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.
    "experiment_0": {
        "model": "model name",
        "hyperparameters": {
            "name_of_hyperparameter_0": "value in str or float - You need to play with at least two different hyperparameters so at least name_of_hyperparameter_0 and name_of_hyperparameter_1",
            "name_of_hyperparameter_1": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description one of the approach - it has to be different for each experiment."
    },
    # Everything in experiment_1 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.
    "experiment_1": {
        "model": "model name",
        "hyperparameters": {
            "name_of_hyperparameter_0": "value in str or float",
            "name_of_hyperparameter_1": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description two of the approach - it has to be different for each experiment."
    },
    # Everything in experiment_2 is related to experiment on validation dataset, so metrics are computed on validation dataset etc.
    "experiment_2": {
        "model": "model name",
        "hyperparameters": {
            "name_of_hyperparameter_0": "value in str or float",
            "name_of_hyperparameter_1": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description three of the approach - it has to be different for each experiment."
    },
    # Everything in final_prediction is related to prediction on test dataset, so metrics are computed on test dataset etc.
    "final_prediction": {
        "model": "google-bert/bert-base-uncased",
        "experiment_chosen": "experiment_0 or experiment_1 or experiment_2",
        "hyperparameters": {
            "name_of_hyperparameter_0": "value in str or float",
            "name_of_hyperparameter_1": "value in str or float"
        },
        "f1_score": "value in float",
        "precision": "value in float",
        "recall": "value in float",
        "description": "Unique description four of the final results and prediction - it has to be different and here you will describe results on test dataset."
    }
}


In [ ]:
data={}

for i, hp_result in enumerate(all_hp_results):
    experiment_key=f"experiment_{i}"

    hyperparameters_for_output={
        "learning_rate": hp_result["hyperparameters"]["learning_rate"],
        "num_train_epochs": hp_result["hyperparameters"]["num_train_epochs"],
        "per_device_train_batch_size": hp_result["hyperparameters"]["per_device_train_batch_size"]
    }

    data[experiment_key] = {
        "model": "distilbert/distilbert-base-uncased",
        "hyperparameters": hyperparameters_for_output,
        "f1_score": hp_result["validation_metrics"]["f1"],
        "precision": hp_result["validation_metrics"]["precision"],
        "recall": hp_result["validation_metrics"]["recall"],
        "description": (
            f"This experiment fine-tuned the distilbert/distilbert-base-uncased model for binary classification "
            f"using a learning rate of {hp_result['hyperparameters']['learning_rate']}, "
            f"batch size {hp_result['hyperparameters']['per_device_train_batch_size']}, "
            f"and {hp_result['hyperparameters']['num_train_epochs']} epochs. "
            f"The model achieved an F1-score of {hp_result['validation_metrics']['f1']:.4f}, "
            f"with precision of {hp_result['validation_metrics']['precision']:.4f} "
            )
    }

# Assuming best_experiment, best_hyperparameters, and evaluation_results are defined elsewhere
data["final_prediction"]={
    "model": "distilbert/distilbert-base-uncased",
    "experiment_chosen": f"experiment_{best_experimenti}",
    "hyperparameters":{
        "learning_rate": best_hyperparameters["learning_rate"],
        "per_device_train_batch_size": best_hyperparameters["per_device_train_batch_size"]
    },
    "f1_score": evaluation_results["f1"],
    "precision": evaluation_results["precision"],
    "recall": evaluation_results["recall"],
    "description": (
        f"Final prediction results on the test dataset using the best model from experiment_{best_experimenti}. "
        f"This model achieved an F1 of {evaluation_results['f1']:.4f}, precision of {evaluation_results['precision']:.4f}, "
        f"and recall of {evaluation_results['recall']:.4f} on the test set. The model was trained with a learning rate of "
        f"{best_hyperparameters['learning_rate']} and batch size of {best_hyperparameters['per_device_train_batch_size']}."
    )
}

In [ ]:
with open("experiments_marta_feltrin_2150892.json", "w") as f:
    json.dump(data, f, indent=4)

## Example final file

In [ ]:
data = {
    "experiment_0": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": {
            "learning_rate": "float",
            "warmap_ratio": "float",
            "weight_decay": "float"
        },
        "f1_score": "float",
        "precision": "float",
        "recall": "float",
        "description": "This experiment fine-tuned the google-bert/bert-base-uncased model for binary classification using a learning rate of 1e-5 and a warmup ratio of 0.06. The model achieved an F1-score of 0.76, with a strong recall of 0.85, indicating high sensitivity to positive cases. Precision was moderate at 0.65, suggesting some trade-off in false positives. The setup demonstrates effective recall-oriented performance in identifying relevant instances."
    },
    "experiment_1": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": {
            "learning_rate": "float",
            "weight_decay": "float"
        },
        "f1_score": "float",
        "precision": "float",
        "recall": "float",
        "description": "Unique description two of the approach - it has to be different for each experiment. Everything in experiment_1 is related to experiment on validation dataset, so metrics are computed on validation dataset etc."
    },
    "experiment_2": {
        "model": "google-bert/bert-base-uncased",
        "hyperparameters": {
            "learning_rate": "float",
            "num_train_epochs": "int",
            "weight_decay": "float"
        },
        "f1_score": "float",
        "precision": "float",
        "recall": "float",
        "description": "Unique description three of the approach - it has to be different for each experiment. Everything in experiment_2 is related to experiment on validation dataset, so metrics are computed on validation dataset etc."
    },
    "final_prediction": {
        "model": "google-bert/bert-base-uncased",
        "experiment_chosen": "experiment_0",
        "hyperparameters": {
            "learning_rate": "float",
            "warmap_ratio": "float"
        },
        "f1_score": "float",
        "precision": "float",
        "recall": "float",
        "description": "Unique description four of the final results and prediction - it has to be different and here you will describe results on test dataset. Everything in final_prediction is related to prediction on test dataset, so metrics are computed on test dataset etc."
    }
}

In [ ]:
with open("experiments_Arkadiusz_Modzelewski_29580.json", "w") as f:
    json.dump(data, f, indent=4)